In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/sales_data.csv")

df

,Order_ID,Order_Date,Customer_ID,Product,Category,Region,Quantity,Unit_Price,Discount,Payment_Method,Returned
0,1001,2026-01-03,C001,Laptop,Electronics,North,1,65000,0.05,UPI,No
1,1002,2026-01-04,C002,Mouse,Accessories,South,2,800,0.00,Credit Card,No
2,1003,2026-01-05,C003,Keyboard,Accessories,West,1,1800,0.10,UPI,No
3,1004,2026-01-06,C004,Monitor,Electronics,East,1,15000,0.05,Debit Card,No
4,1005,2026-01-08,C005,Headphones,Accessories,North,2,2500,0.00,UPI,Yes
5,1006,2026-01-10,C006,Laptop,Electronics,South,1,72000,0.08,Credit Card,No
6,1007,2026-01-11,C007,Mouse,Accessories,West,3,750,0.00,Cash,No
7,1008,2026-01-13,C008,Monitor,Electronics,East,2,14000,0.10,UPI,No
8,1009,2026-01-15,C009,Keyboard,Accessories,North,2,1900,0.05,Debit Card,No
9,1010,2026-01-16,C010,Headphones,Accessories,South,1,2800,0.00,UPI,Yes


In [3]:
messy_df = df.copy()

In [5]:
messy_df.loc[2, "Category"] = "accessories"
messy_df.loc[5, "Product"] = None
messy_df.loc[8, "Quantity"] = -2

messy_df = pd.concat([messy_df, messy_df.iloc[[3]]], ignore_index=True)
"""Now we have:

1) inconsistent category capitalization
2) missing product
3) invalid negative quantity
4) duplicate order"""

'Now we have:\n\n1) inconsistent category capitalization\n2) missing product\n3) invalid negative quantity\n4) duplicate order'

In [6]:
# Data Quality Audit
print("Rows:", messy_df.shape[0])
print("Columns:", messy_df.shape[1])

Rows: 17
Columns: 11


In [7]:
messy_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order_ID        17 non-null     int64  
 1   Order_Date      17 non-null     object 
 2   Customer_ID     17 non-null     object 
 3   Product         16 non-null     object 
 4   Category        17 non-null     object 
 5   Region          17 non-null     object 
 6   Quantity        17 non-null     int64  
 7   Unit_Price      17 non-null     int64  
 8   Discount        17 non-null     float64
 9   Payment_Method  17 non-null     object 
 10  Returned        17 non-null     object 
dtypes: float64(1), int64(3), object(7)
memory usage: 1.6+ KB


In [8]:
#Missing values
messy_df.isnull().sum()

Order_ID          0
Order_Date        0
Customer_ID       0
Product           1
Category          0
Region            0
Quantity          0
Unit_Price        0
Discount          0
Payment_Method    0
Returned          0
dtype: int64

In [9]:
# Duplicate records
messy_df.duplicated().sum()

np.int64(2)

In [10]:
# Invalid quantities
messy_df[messy_df["Quantity"] <= 0]

,Order_ID,Order_Date,Customer_ID,Product,Category,Region,Quantity,Unit_Price,Discount,Payment_Method,Returned
8,1009,2026-01-15,C009,Keyboard,Accessories,North,-2,1900,0.05,Debit Card,No


In [11]:
# Clean the missing value
cleaned_df = messy_df.dropna(subset=["Product"]).copy()

In [12]:
cleaned_df.isnull().sum()

Order_ID          0
Order_Date        0
Customer_ID       0
Product           0
Category          0
Region            0
Quantity          0
Unit_Price        0
Discount          0
Payment_Method    0
Returned          0
dtype: int64

In [13]:
#Remove duplicates
cleaned_df = cleaned_df.drop_duplicates()

In [14]:
cleaned_df.duplicated().sum()

np.int64(0)

In [15]:
# Fix invalid quantities
cleaned_df[cleaned_df["Quantity"] <= 0]

,Order_ID,Order_Date,Customer_ID,Product,Category,Region,Quantity,Unit_Price,Discount,Payment_Method,Returned
8,1009,2026-01-15,C009,Keyboard,Accessories,North,-2,1900,0.05,Debit Card,No


In [ ]:
# Then remove them:
cleaned_df = cleaned_df[cleaned_df["Quantity"] > 0]

In [18]:
#Check:
cleaned_df["Quantity"].min()
# It should now be greater than 0

1

In [ ]:
# Standardize Category names 
# we currently have 1) Accessories 2) accessories
cleaned_df["Category"] = cleaned_df["Category"].str.strip().str.title()

In [20]:
cleaned_df["Category"].unique()

array(['Electronics', 'Accessories'], dtype=object)

In [21]:
#Fix the date column
# Currently Pandas may treat Order_Date as an object/string.
# Now we will convert it

cleaned_df["Order_Date"] = pd.to_datetime(cleaned_df["Order_Date"])

In [22]:
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13 entries, 0 to 14
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Order_ID        13 non-null     int64         
 1   Order_Date      13 non-null     datetime64[ns]
 2   Customer_ID     13 non-null     object        
 3   Product         13 non-null     object        
 4   Category        13 non-null     object        
 5   Region          13 non-null     object        
 6   Quantity        13 non-null     int64         
 7   Unit_Price      13 non-null     int64         
 8   Discount        13 non-null     float64       
 9   Payment_Method  13 non-null     object        
 10  Returned        13 non-null     object        
dtypes: datetime64[ns](1), float64(1), int64(3), object(6)
memory usage: 1.2+ KB


In [23]:
# Recalculate Sales
cleaned_df["Sales"] = (
    cleaned_df["Quantity"] * cleaned_df["Unit_Price"]
)

In [24]:
# Final Data Quality Check
print("Missing values:")
print(cleaned_df.isnull().sum())

print("\nDuplicate rows:")
print(cleaned_df.duplicated().sum())

print("\nInvalid quantities:")
print((cleaned_df["Quantity"] <= 0).sum())

Missing values:
Order_ID          0
Order_Date        0
Customer_ID       0
Product           0
Category          0
Region            0
Quantity          0
Unit_Price        0
Discount          0
Payment_Method    0
Returned          0
Sales             0
dtype: int64

Duplicate rows:
0

Invalid quantities:
0


In [25]:
# Compare Before vs After
print("Rows before cleaning:", messy_df.shape[0])
print("Rows after cleaning:", cleaned_df.shape[0])

print("Rows removed:", messy_df.shape[0] - cleaned_df.shape[0])

Rows before cleaning: 17
Rows after cleaning: 13
Rows removed: 4


In [26]:
#Save the cleaned dataset
cleaned_df.to_csv(
    "../data/cleaned/sales_data_cleaned.csv",
    index=False
)

## Day 2 — Data Cleaning Summary

### Issues Identified

- Missing product values
- Duplicate records
- Invalid negative quantities
- Inconsistent category capitalization
- Order date stored as text

### Cleaning Performed

- Removed rows with missing products
- Removed duplicate records
- Removed invalid quantities
- Standardized category names
- Converted Order_Date to datetime
- Recalculated Sales

### Validation

After cleaning:

- Missing values: ___
- Duplicate records: ___
- Invalid quantities: ___
- Rows before cleaning: ___
- Rows after cleaning: ___

The cleaned dataset has been exported to the `data/cleaned/` directory.